# Flight selection model evaluation

Run direct Duffel + TypeSafe selection with one or more Jev models against a fixed flight corpus. The legacy MCP/subagent baseline is optional. Each case is run sequentially so another live search cannot distort its timing.

> Live experiment: every run makes real provider calls and may incur cost.

In [26]:
from __future__ import annotations

import asyncio
import json
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from time import perf_counter

from dotenv import load_dotenv
from langchain.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langchain_typesafe import TypeSafeClassifier
from pydantic import BaseModel, Field

# The notebook lives two levels below the repository root.
EXPERIMENT_DIR = Path.cwd().resolve()
if not (EXPERIMENT_DIR / 'flight_evaluation_state.json').exists():
    EXPERIMENT_DIR = Path.cwd().resolve() / 'experiments' / 'flight_evaluation'
REPO_ROOT = EXPERIMENT_DIR.parents[1]
sys.path.insert(0, str(REPO_ROOT))       # preserved /legacy benchmark package
sys.path.insert(0, str(REPO_ROOT / 'src'))  # active application package

from travel_agent.core.models import BestOffer, FlightPipelineMetric, Leg
from travel_agent.flights.duffel import DuffelClient
from travel_agent.flights.flight_selector import (
    baggage_requested, enrich_baggage_details, parse_duffel_offers,
    prepare_baggage_candidates, prepare_candidates, select_best_offer,
)
from travel_agent.infrastructure.resources import open_resources

load_dotenv(REPO_ROOT / '.env')


True

In [27]:
# Edit this cell before each experiment.
JEV_MODELS = ['jev-1.13.0']
RUN_LEGACY_BASELINE = True
LEGACY_MODELS = ['openai:gpt-5-mini', 'openai:gpt-6-luna']  # Each model runs the preserved legacy pipeline.
# Chat Completions supports Luna function tools only with reasoning disabled.
LEGACY_MODEL_KWARGS = {'openai:gpt-6-luna': {'reasoning_effort': 'none'}}
CASE_IDS: set[int] | None = None  # e.g. {1, 4}; None runs every case
STATE_PATH = EXPERIMENT_DIR / 'flight_evaluation_state.json'
OUTPUT_PATH = EXPERIMENT_DIR / 'model_comparison_results.json'

# USD per one million tokens. Add the exact provider price before comparing a new model.
INPUT_TOKEN_PRICES = {'jev-1.13.0': 0.042, 'openai:gpt-5-mini': 0.25, 'openai:gpt-6-luna': 0.1}
OUTPUT_TOKEN_PRICES = {'jev-1.13.0': 0.0, 'openai:gpt-5-mini': 2.0, 'openai:gpt-6-luna': 0.5}


In [28]:
class FlightEvaluationState(BaseModel):
    legs: list[Leg] = Field(min_length=1)
    travelers: int = Field(ge=1)
    flight_preferences: str | None = None

class FlightEvaluationCase(BaseModel):
    id: int = Field(ge=1)
    prompt: str = Field(min_length=1)
    state: FlightEvaluationState

class FlightEvaluationCorpus(BaseModel):
    test_states: list[FlightEvaluationCase] = Field(min_length=1)

def estimated_cost(model: str, input_tokens: int | None, output_tokens: int | None) -> float | None:
    if input_tokens is None or output_tokens is None:
        return None
    if model not in INPUT_TOKEN_PRICES or model not in OUTPUT_TOKEN_PRICES:
        return None
    return round((input_tokens * INPUT_TOKEN_PRICES[model] + output_tokens * OUTPUT_TOKEN_PRICES[model]) / 1_000_000, 8)

def legacy_model_client(model: str):
    """Create model-specific legacy clients while retaining string models by default."""
    provider, model_name = model.split(':', 1)
    kwargs = LEGACY_MODEL_KWARGS.get(model)
    if provider == 'openai' and kwargs is not None:
        return ChatOpenAI(model=model_name, **kwargs)
    return model

def aggregate_usage(metrics: list[FlightPipelineMetric]) -> tuple[int | None, int | None]:
    """Do not turn a failed or usage-less run into a false zero-token result."""
    if any(metric.error or metric.input_tokens is None or metric.output_tokens is None for metric in metrics):
        return None, None
    return sum(metric.input_tokens for metric in metrics), sum(metric.output_tokens for metric in metrics)

def selected(leg_index: int, leg: Leg, offer: BestOffer | None, error: str | None = None) -> dict:
    return {
        'leg_index': leg_index, 'origin': leg.origin, 'destination': leg.destination,
        'offer_id': offer.offer_id if offer else None, 'price': offer.price if offer else None,
        'currency': offer.currency if offer else None, 'reasoning': offer.reasoning if offer else None,
        'departure': offer.departure if offer else None, 'arrival': offer.arrival if offer else None,
        'duration_minutes': offer.duration_minutes if offer else None, 'stops': offer.stops if offer else None,
        'carriers': offer.carriers if offer else [],
        'included_checked_baggage': offer.included_checked_baggage if offer else None,
        'additional_checked_baggage': offer.additional_checked_baggage if offer else [],
        'error': error,
    }

corpus = FlightEvaluationCorpus.model_validate_json(STATE_PATH.read_text())
cases = [case for case in corpus.test_states if CASE_IDS is None or case.id in CASE_IDS]
assert cases, 'No cases match CASE_IDS.'
print(f'Loaded {len(cases)} case(s): {[case.id for case in cases]}')


Loaded 10 case(s): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [29]:
async def evaluate_jev_case(case: FlightEvaluationCase, model: str) -> dict:
    """Run one direct-Duffel/Jev model experiment for every leg in a case."""
    started = perf_counter()
    client = DuffelClient.from_environment()
    classifier = TypeSafeClassifier(model=model)
    semaphore = asyncio.Semaphore(3)

    async def evaluate_leg(index: int, leg: Leg):
        leg_started = perf_counter()
        duffel_ms = baggage_ms = selection_ms = None
        candidates = 0
        try:
            duffel_started = perf_counter()
            async with semaphore:
                request = await client.search_one_way(
                    origin=leg.origin, destination=leg.destination,
                    departure_date=leg.departure_date or '', travelers=case.state.travelers,
                )
            duffel_ms = round((perf_counter() - duffel_started) * 1000)
            options = prepare_candidates(parse_duffel_offers(request))
            if baggage_requested(case.state.flight_preferences):
                baggage_started = perf_counter()
                options = await enrich_baggage_details(client, prepare_baggage_candidates(options))
                baggage_ms = round((perf_counter() - baggage_started) * 1000)
            candidates = len(options)
            if not options:
                raise ValueError('Duffel returned no viable offers.')
            selection_started = perf_counter()
            offer, confidence, input_tokens, output_tokens, response_model = await select_best_offer(
                classifier, origin=leg.origin, destination=leg.destination,
                departure_date=leg.departure_date or '', preferences=case.state.flight_preferences,
                candidates=options,
            )
            selection_ms = round((perf_counter() - selection_started) * 1000)
            metric = FlightPipelineMetric(
                pipeline='jev', leg_index=index, duffel_ms=duffel_ms,
                baggage_enrichment_ms=baggage_ms, selection_ms=selection_ms,
                total_ms=round((perf_counter() - leg_started) * 1000), candidates=candidates,
                selected_offer_id=offer.offer_id, selection_confidence=confidence,
                input_tokens=input_tokens, output_tokens=output_tokens, model=response_model,
            )
            return selected(index, leg, offer), metric
        except Exception as exc:
            metric = FlightPipelineMetric(
                pipeline='jev', leg_index=index, duffel_ms=duffel_ms,
                baggage_enrichment_ms=baggage_ms, selection_ms=selection_ms,
                total_ms=round((perf_counter() - leg_started) * 1000), candidates=candidates, error=str(exc),
            )
            return selected(index, leg, None, str(exc)), metric

    try:
        results = await asyncio.gather(*(evaluate_leg(index, leg) for index, leg in enumerate(case.state.legs)))
    finally:
        await client.aclose()
        if classifier.async_client is not None:
            await classifier.async_client.aclose()
        if classifier.client is not None:
            classifier.client.close()
    flights, metrics = map(list, zip(*results))
    input_tokens, output_tokens = aggregate_usage(metrics)
    response_model = next((metric.model for metric in metrics if metric.model), model)
    error = next((metric.error for metric in metrics if metric.error), None)
    return {
        'model': response_model, 'end_to_end_ms': round((perf_counter() - started) * 1000),
        'input_tokens': input_tokens, 'output_tokens': output_tokens,
        'total_token_cost_usd': estimated_cost(response_model, input_tokens, output_tokens),
        'flights': flights, 'metrics': [metric.model_dump() for metric in metrics],
        'error': error, 'status': 'error' if error else 'ok',
    }


In [30]:
async def evaluate_legacy_case(case: FlightEvaluationCase, model: str) -> dict:
    """Optional preserved MCP/subagent baseline."""
    started = perf_counter()
    old_pipeline = os.environ.get('FLIGHT_PIPELINE')
    os.environ['FLIGHT_PIPELINE'] = 'legacy'
    try:
        setup_started = perf_counter()
        async with open_resources(legacy_model=legacy_model_client(model)) as resources:
            setup_ms = round((perf_counter() - setup_started) * 1000)
            semaphore = asyncio.Semaphore(3)

            async def evaluate_leg(index: int, leg: Leg):
                leg_started = perf_counter()
                message = (
                    f'{case.prompt}\n\nFind the best one-way flight from {leg.origin} to {leg.destination} '
                    f'departing on {leg.departure_date}.\n\nTraveller preferences: '
                    f"{case.state.flight_preferences or 'No saved flight preferences.'}"
                )
                try:
                    async with semaphore:
                        result = await resources.flights_agent.ainvoke({'messages': [HumanMessage(content=message)]})
                    usage = [getattr(item, 'usage_metadata', None) or {} for item in result.get('messages', [])]
                    usage_records = [item for item in usage if isinstance(item, dict) and ('input_tokens' in item or 'output_tokens' in item)]
                    input_tokens = sum(int(item.get('input_tokens') or 0) for item in usage_records) if usage_records else None
                    output_tokens = sum(int(item.get('output_tokens') or 0) for item in usage_records) if usage_records else None
                    offer = result.get('structured_response')
                    metric = FlightPipelineMetric(
                        pipeline='legacy', leg_index=index, total_ms=round((perf_counter() - leg_started) * 1000),
                        selected_offer_id=offer.offer_id if offer else None, model=model,
                        input_tokens=input_tokens, output_tokens=output_tokens,
                    )
                    return selected(index, leg, offer), metric
                except Exception as exc:
                    metric = FlightPipelineMetric(pipeline='legacy', leg_index=index, total_ms=round((perf_counter() - leg_started) * 1000), error=str(exc))
                    return selected(index, leg, None, str(exc)), metric

            results = await asyncio.gather(*(evaluate_leg(index, leg) for index, leg in enumerate(case.state.legs)))
    finally:
        if old_pipeline is None:
            os.environ.pop('FLIGHT_PIPELINE', None)
        else:
            os.environ['FLIGHT_PIPELINE'] = old_pipeline
    flights, metrics = map(list, zip(*results))
    input_tokens, output_tokens = aggregate_usage(metrics)
    error = next((metric.error for metric in metrics if metric.error), None)
    return {
        'model': model, 'setup_ms': setup_ms,
        'end_to_end_ms': round((perf_counter() - started) * 1000),
        'input_tokens': input_tokens, 'output_tokens': output_tokens,
        'total_token_cost_usd': estimated_cost(model, input_tokens, output_tokens),
        'flights': flights, 'metrics': [metric.model_dump() for metric in metrics],
        'error': error, 'status': 'error' if error else 'ok',
    }


In [31]:
async def run_experiment() -> dict:
    results = []
    for case in cases:
        print(f'Case {case.id}: {case.state.legs[0].origin} → {case.state.legs[0].destination}')
        model_results = {}
        for model in JEV_MODELS:
            print(f'  Jev: {model}', flush=True)
            model_results[model] = await evaluate_jev_case(case, model)
        legacy_results = {}
        if RUN_LEGACY_BASELINE:
            for model in LEGACY_MODELS:
                print(f'  legacy: {model}', flush=True)
                legacy_results[model] = await evaluate_legacy_case(case, model)
        results.append({'id': case.id, 'prompt': case.prompt, 'jev_models': model_results, 'legacy_models': legacy_results})
        OUTPUT_PATH.write_text(json.dumps({'results': results}, indent=2) + '\n')
    return {'results': results}

results = await run_experiment()
print(f'Saved {len(results["results"])} case(s) to {OUTPUT_PATH}')


Case 1: LHR → CDG
  Jev: jev-1.13.0
  legacy: openai:gpt-5-mini
  legacy: openai:gpt-6-luna
Case 2: JFK → LAX
  Jev: jev-1.13.0
  legacy: openai:gpt-5-mini
  legacy: openai:gpt-6-luna
Case 3: SFO → SEA
  Jev: jev-1.13.0
  legacy: openai:gpt-5-mini
  legacy: openai:gpt-6-luna
Case 4: LAX → JFK
  Jev: jev-1.13.0
  legacy: openai:gpt-5-mini
  legacy: openai:gpt-6-luna
Case 5: CDG → FCO
  Jev: jev-1.13.0
  legacy: openai:gpt-5-mini
  legacy: openai:gpt-6-luna
Case 6: NRT → ICN
  Jev: jev-1.13.0
  legacy: openai:gpt-5-mini
  legacy: openai:gpt-6-luna
Case 7: SYD → MEL
  Jev: jev-1.13.0
  legacy: openai:gpt-5-mini
  legacy: openai:gpt-6-luna
Case 8: BOS → MIA
  Jev: jev-1.13.0
  legacy: openai:gpt-5-mini
  legacy: openai:gpt-6-luna
Case 9: AMS → BCN
  Jev: jev-1.13.0
  legacy: openai:gpt-5-mini
  legacy: openai:gpt-6-luna
Case 10: ORD → DEN
  Jev: jev-1.13.0
  legacy: openai:gpt-5-mini
  legacy: openai:gpt-6-luna
Saved 10 case(s) to /Users/caedinm/Projects/travel_agent/experiments/flight_eva

In [32]:
# Case-by-case comparison. `N/A` means the run failed or reported no usage.
def display_usage(result: dict) -> str:
    if result['input_tokens'] is None or result['output_tokens'] is None:
        return 'N/A (failed or usage unavailable)'
    return f"{result['input_tokens']} in | {result['output_tokens']} out | ${result['total_token_cost_usd']}"

def format_duration(minutes: int | None) -> str:
    if minutes is None:
        return 'duration unavailable'
    return f'{minutes // 60}h {minutes % 60}m'

def display_selected_flights(result: dict) -> None:
    for flight in result['flights']:
        route = f"{flight['origin']} → {flight['destination']}"
        if flight.get('error'):
            print(f"      {route}: no selection — {flight['error']}")
            continue
        price = f"{flight.get('currency') or ''} {flight.get('price') or 'unavailable'}".strip()
        carriers = ', '.join(flight.get('carriers') or []) or 'carrier unavailable'
        stops = flight.get('stops')
        stop_label = 'stops unavailable' if stops is None else ('nonstop' if stops == 0 else f'{stops} stop(s)')
        schedule = ' → '.join(value for value in [flight.get('departure'), flight.get('arrival')] if value)
        print(f"      {route}: {price} | {carriers} | {stop_label} | {format_duration(flight.get('duration_minutes'))}")
        if schedule:
            print(f"        Schedule: {schedule}")
        if flight.get('included_checked_baggage'):
            print(f"        Checked bag: {flight['included_checked_baggage']}")
        if flight.get('additional_checked_baggage'):
            print(f"        Extra bag options: {', '.join(flight['additional_checked_baggage'])}")
        print(f"        Offer: {flight.get('offer_id') or 'unavailable'} — {flight.get('reasoning') or 'No rationale returned.'}")

for case in results['results']:
    print(f"\nCase {case['id']}")
    for model, result in case['jev_models'].items():
        print(f"  {model}: {result['end_to_end_ms']} ms | {display_usage(result)}")
        display_selected_flights(result)
    for model, result in case['legacy_models'].items():
        print(f"  legacy / {model}: {result['end_to_end_ms']} ms | {display_usage(result)}")
        display_selected_flights(result)



Case 1
  jev-1.13.0: 2562 ms | 16655 in | 950 out | $0.00069951
      LHR → CDG: USD 41.93 | Duffel Airways | nonstop | 1h 2m
        Schedule: 2026-10-15T10:50:00 → 2026-10-15T12:52:00
        Checked bag: 1 checked bag included per passenger
        Offer: off_0000BAggAblCtPfrUNP55J — Selected by Jev from 39 viable options with 100% decision confidence.
  legacy / openai:gpt-5-mini: 14203 ms | 2798 in | 843 out | $0.0023855
      LHR → CDG: USD 42.38 | carrier unavailable | stops unavailable | duration unavailable
        Offer: off_0000BAggB59w50pkVcevzg — Chose this offer because it is by far the lowest fare (USD 42.38) and is a non-stop departing at a civilised 10:50, so it avoids early-morning departures and connections for only a tiny saving.
  legacy / openai:gpt-6-luna: 7505 ms | 2801 in | 146 out | $0.0003531
      LHR → CDG: USD 41.82 | Duffel Airways | nonstop | 1h 2m
        Schedule: 10:50 → 12:52
        Offer: off_0000BAggCUSozZUFREhYC7 — The $41.82 nonstop at 10:50 av

In [33]:
import pandas as pd

rows = []
for case in results['results']:
    for model, result in case['jev_models'].items():
        rows.append({'pipeline': 'jev', 'model': model, **result})
    for model, result in case['legacy_models'].items():
        rows.append({'pipeline': 'legacy', 'model': model, **result})

raw_results = pd.DataFrame(rows)
comparison = (
    raw_results.groupby(['pipeline', 'model'], as_index=False)
    .agg(
        cases=('end_to_end_ms', 'size'),
        successful_cases=('status', lambda values: (values == 'ok').sum()),
        failed_cases=('status', lambda values: (values == 'error').sum()),
        average_end_to_end_ms=('end_to_end_ms', 'mean'),
        total_input_tokens=('input_tokens', lambda values: values.sum(min_count=1)),
        total_output_tokens=('output_tokens', lambda values: values.sum(min_count=1)),
        estimated_total_cost_usd=('total_token_cost_usd', lambda values: values.sum(min_count=1)),
    )
    .sort_values(['pipeline', 'model'])
)
comparison['average_end_to_end_ms'] = comparison['average_end_to_end_ms'].round().astype('Int64')
comparison['total_input_tokens'] = comparison['total_input_tokens'].round().astype('Int64')
comparison['total_output_tokens'] = comparison['total_output_tokens'].round().astype('Int64')
comparison['estimated_total_cost_usd'] = comparison['estimated_total_cost_usd'].round(8)
comparison


,pipeline,model,cases,successful_cases,failed_cases,average_end_to_end_ms,total_input_tokens,total_output_tokens,estimated_total_cost_usd
0,jev,jev-1.13.0,10,10,0,2657,90211,5273,0.003789
1,legacy,openai:gpt-5-mini,10,10,0,18302,27160,10291,0.027372
2,legacy,openai:gpt-6-luna,10,10,0,6722,25437,1197,0.003142
